# Monitor Financial-GPT — 4 modelos globales + 3 baselines

Compara exclusivamente las cuatro arquitecturas globales sobre las mismas fechas de test por `cross_key_id`. Usa los mismos baselines que `monitor_codigo_02.ipynb`: `NAIVE_LAST_VALUE`, `NAIVE_ZERO` sólo para variación y `SEASONAL_NAIVE_FINANCIAL`. Los modelos locales permanecen separados.

In [ ]:
# Parámetros
AUTO_DISCOVER_GLOBAL = True

GLOBAL_MODEL_ROOT = "s3://your-private-bucket/users/your-user/financial_gpt"

# Úsalos sólo cuando quieras fijar runs concretos.
GLOBAL_RUN_URIS = {
    # "GLOBAL_MLP_E_D": "s3://.../financial_gpt/mlp/runs/...",
    # "GLOBAL_MLP_VaE_D": "s3://.../financial_gpt/mlp_vae/runs/...",
    # "GLOBAL_RNN_E_D": "s3://.../financial_gpt/rnn/runs/...",
    # "GLOBAL_RNNBi_E_D": "s3://.../financial_gpt/rnn_bi/runs/...",
}

WINNER_METRICS = ["MASE"]  # único criterio causal y no redundante
INCLUDE_BASELINES = True
SEASONAL_PERIOD_DAYS = 7
OUTPUT_DIRECTORY = "./financial_gpt_global_monitor_output"

In [ ]:
from IPython.display import display

from financial_gpt_monitor import (
    compare_global_financial_gpt_runs,
    discover_latest_global_runs,
)


## 1. Resolver los cuatro runs globales


In [ ]:
global_runs = (
    discover_latest_global_runs(GLOBAL_MODEL_ROOT)
    if AUTO_DISCOVER_GLOBAL
    else dict(GLOBAL_RUN_URIS)
)

print("Runs globales:")
for name, uri in global_runs.items():
    print(" ", name, "->", uri)


## 2. Comparación común y ganador global por serie


In [ ]:
monitor = compare_global_financial_gpt_runs(
    global_run_uris=global_runs,
    metrics=WINNER_METRICS,
    include_baselines=INCLUDE_BASELINES,
    seasonal_period_days=SEASONAL_PERIOD_DAYS,
)

display(monitor.run_inventory)
display(monitor.comparison_coverage)
display(monitor.winner_counts)
display(monitor.winners_by_series)
display(monitor.ensemble_forecast.head(50))

## 3. Guardar salidas y gates


In [ ]:
output = monitor.write(OUTPUT_DIRECTORY)

assert monitor.run_inventory.height == 7
assert monitor.winners_by_series.height > 0
assert monitor.ensemble_forecast.height > 0
assert set(monitor.run_inventory["family"].unique()).issubset({"global", "baseline"})
assert set(monitor.winners_by_series["winner_family"].unique()).issubset({"global", "baseline"})
assert monitor.ensemble_forecast["cross_key_id"].n_unique() <= monitor.winners_by_series.height

assert output.name == "financial_gpt_monitor.json"
assert list(output.parent.glob("*.parquet")) == []
print("JSON único del monitor:", output)
print("Candidatos: 4 globales + 3 baselines")
print("Baselines: NAIVE_LAST_VALUE, NAIVE_ZERO para variación, SEASONAL_NAIVE_FINANCIAL")
print("Periodo estacional:", SEASONAL_PERIOD_DAYS, "días")
print("Series comparadas:", monitor.winners_by_series.height)
print("Filas del ensemble:", monitor.ensemble_forecast.height)
print("✅ ningún modelo local entra a este monitor")
print("✅ métricas recalculadas sobre fechas comunes de test")
print("✅ ganador global/baseline y forecast final por cross_key_id")